# Hopf fibration: from product state to Bell state

Visualises how quantum information migrates from **local** (Bloch sphere content) to **correlations** (Hopf base S² + fibre S¹) as two qubits become entangled.

$$|\psi\rangle = \cos\theta\,|00\rangle + \sin\theta\,|11\rangle$$

- **Bloch vector length** = $|\cos 2\theta|$ (shrinks to zero at maximal entanglement)
- **Concurrence** = $\sin 2\theta$ (grows to one)
- Information is conserved — it moves from local to fibre

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, Arc
from matplotlib.collections import PatchCollection
import ipywidgets as widgets
from IPython.display import display

In [ ]:
def draw_bloch_sphere(ax, centre, radius, bloch_len, label):
    """Draw a 2D projection of a Bloch sphere with state vector."""
    ax.add_patch(Circle(centre, radius, fill=False, ec='#888780', lw=0.8))

    # equator ellipse
    eq = Arc(centre, 2 * radius, 2 * radius * 0.35, angle=0,
             theta1=0, theta2=360, ls='--', lw=0.6, color='#888780')
    ax.add_patch(eq)

    # shaded accessible region
    if bloch_len > 0.01:
        ax.add_patch(Circle(centre, radius * bloch_len,
                            fc='#5DCAA5', alpha=0.2, ec='none'))

    # Bloch vector
    cx, cy = centre
    if bloch_len > 0.02:
        vec_len = bloch_len * radius
        ax.annotate('', xy=(cx, cy + vec_len), xytext=(cx, cy),
                    arrowprops=dict(arrowstyle='->', color='#1D9E75', lw=2.2))
    else:
        ax.plot(cx, cy, 'o', color='#1D9E75', ms=5, zorder=5)

    # pole labels
    ax.text(cx, cy + radius + 0.06, '|0⟩', ha='center', va='bottom',
            fontsize=9, color='#888780')
    ax.text(cx, cy - radius - 0.06, '|1⟩', ha='center', va='top',
            fontsize=9, color='#888780')
    ax.text(cx, cy - radius - 0.22, label, ha='center', va='top',
            fontsize=10, color='#2C2C2A', fontweight='medium')


def draw_hopf_base(ax, centre, radius, concurrence):
    """Draw the Hopf base S² with entanglement class dot."""
    ax.add_patch(Circle(centre, radius, fill=False, ec='#888780', lw=0.8))
    eq = Arc(centre, 2 * radius, 2 * radius * 0.35, angle=0,
             theta1=0, theta2=360, ls='--', lw=0.6, color='#888780')
    ax.add_patch(eq)

    cx, cy = centre
    dot_y = cy + radius - concurrence * 2 * radius  # top = product, bottom = Bell

    # glow
    if concurrence > 0.02:
        glow_r = 0.04 + concurrence * 0.06
        ax.add_patch(Circle((cx, dot_y), glow_r,
                            fc='#7F77DD', alpha=0.2, ec='none'))

    dot_r = 0.02 + concurrence * 0.03
    ax.plot(cx, dot_y, 'o', color='#7F77DD', ms=4 + concurrence * 5, zorder=5)

    ax.text(cx, cy + radius + 0.06, 'Product', ha='center', va='bottom',
            fontsize=9, color='#888780')
    ax.text(cx, cy - radius - 0.06, 'Bell', ha='center', va='top',
            fontsize=9, color='#888780')
    ax.text(cx, cy - radius - 0.22, 'Base S²', ha='center', va='top',
            fontsize=10, color='#2C2C2A', fontweight='medium')


def draw_fibre(ax, centre, radius, concurrence, phase_angle):
    """Draw the S¹ fibre circle with phase marker."""
    alpha = max(0.15, concurrence)
    ax.add_patch(Circle(centre, radius, fill=False,
                        ec='#D85A30', lw=2, alpha=alpha))

    cx, cy = centre
    if concurrence > 0.02:
        px = cx + radius * np.cos(phase_angle)
        py = cy + radius * np.sin(phase_angle)
        ax.plot(px, py, 'o', color='#D85A30', ms=5, alpha=alpha, zorder=5)

    ax.text(cx, cy - radius - 0.06, 'Fibre S¹', ha='center', va='top',
            fontsize=10, color='#2C2C2A' if concurrence > 0.3 else '#B4B2A9',
            fontweight='medium')
    ax.text(cx, cy - radius - 0.20, 'Phase (unobservable)', ha='center',
            va='top', fontsize=8, color='#888780', alpha=alpha)


def draw_info_bar(ax_bar, local_frac):
    """Draw the local vs correlation information partition bar."""
    ax_bar.clear()
    ax_bar.set_xlim(0, 1)
    ax_bar.set_ylim(0, 1)
    ax_bar.set_axis_off()

    h = 0.35
    y0 = 0.3
    corr_frac = 1 - local_frac

    if local_frac > 0.005:
        ax_bar.barh(y0, local_frac, height=h, color='#5DCAA5', left=0)
    if corr_frac > 0.005:
        ax_bar.barh(y0, corr_frac, height=h, color='#7F77DD', left=local_frac)

    ax_bar.add_patch(plt.Rectangle((0, y0), 1, h, fill=False,
                                    ec='#888780', lw=0.5))

    ax_bar.text(0, y0 + h + 0.12,
                f'Local: {local_frac*100:.0f}%',
                fontsize=9, color='#1D9E75', ha='left')
    ax_bar.text(1, y0 + h + 0.12,
                f'Correlations: {corr_frac*100:.0f}%',
                fontsize=9, color='#534AB7', ha='right')

    ax_bar.text(0.25, y0 - 0.18, '● Local (Bloch)',
                fontsize=8, color='#1D9E75', ha='center')
    ax_bar.text(0.75, y0 - 0.18, '● Base S² + Fibre S¹',
                fontsize=8, color='#534AB7', ha='center')

In [ ]:
fig = plt.figure(figsize=(9, 5.5), constrained_layout=True)
gs = fig.add_gridspec(2, 1, height_ratios=[4, 1])
ax = fig.add_subplot(gs[0])
ax_bar = fig.add_subplot(gs[1])

phase_angle = [0.0]  # mutable for animation


def update(pct):
    theta = (pct / 100) * (np.pi / 4)
    bloch_len = abs(np.cos(2 * theta))
    concurrence = np.sin(2 * theta)

    ax.clear()
    ax.set_xlim(-0.1, 1.6)
    ax.set_ylim(-0.55, 0.65)
    ax.set_aspect('equal')
    ax.set_axis_off()

    # state label
    if pct == 0:
        label = r'$|\psi\rangle = |00\rangle$'
    elif pct == 100:
        label = r'$|\psi\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$'
    else:
        c, s = np.cos(theta), np.sin(theta)
        label = rf'$|\psi\rangle = {c:.2f}\,|00\rangle + {s:.2f}\,|11\rangle$'
    ax.set_title(label, fontsize=12, pad=8, family='serif')

    r = 0.25
    draw_bloch_sphere(ax, (0.15, 0.0), r, bloch_len, 'Qubit A')
    draw_bloch_sphere(ax, (0.65, 0.0), r, bloch_len, 'Qubit B')
    draw_hopf_base(ax, (1.10, 0.0), r, concurrence)
    draw_fibre(ax, (1.45, 0.12), 0.15, concurrence, phase_angle[0])

    ax.text(0.40, 0.55, 'Local states', ha='center', fontsize=10,
            color='#5F5E5A')
    ax.text(1.27, 0.55, 'Hopf decomposition', ha='center', fontsize=10,
            color='#5F5E5A')

    # separator line
    ax.axvline(0.88, color='#D3D1C7', lw=0.5, ls='--', ymin=0.1, ymax=0.9)

    # purity annotation
    ax.text(0.40, -0.50, f'Purity: {bloch_len*100:.0f}%',
            ha='center', fontsize=9, color='#888780')
    ax.text(1.27, -0.50, f'Concurrence: {concurrence*100:.0f}%',
            ha='center', fontsize=9, color='#888780')

    draw_info_bar(ax_bar, bloch_len)
    fig.canvas.draw_idle()


slider = widgets.IntSlider(
    value=0, min=0, max=100, step=1,
    description='Entanglement:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%'),
    continuous_update=True
)

widgets.interactive(update, pct=slider)
display(slider)
update(0)

## What you're seeing

**Left — local Bloch spheres.** Each qubit's reduced density matrix has Bloch vector length $|\cos 2\theta|$. At zero entanglement the vectors are full-length (pure local states). At maximal entanglement they collapse to zero — the local states are maximally mixed.

**Right — Hopf base and fibre.** The purple dot on the base $S^2$ tracks the entanglement class (product pole → Bell pole). The coral $S^1$ fibre carries the relative phase that becomes a gauge degree of freedom — exactly the U(1) phase that FANOUT forbids cloning.

**Bottom bar — information conservation.** Total information is constant; it migrates from local (green) to correlations (purple). This is the Cayley–Dickson qubit's internal degrees of freedom ceasing to be locally accessible and becoming the gauge/topological content of the fibration.